# DATA PESANAN SHOPEE


Menggabungkan data pesanan Shopee

In [ ]:
import pandas as pd
import glob
import os

path_folder ="/content/drive/MyDrive/Data Pesanan/Shopee/DES 25-MEI 26"
files = glob.glob(os.path.join(path_folder, "*.xlsx"))

print(f"Ditemukan {len(files)} file di Google Drive. Memulai proses penggabungan...")

all_data = []

for file in files:
    df = pd.read_excel(file)
    df = df.dropna(how='all')
    all_data.append(df)

df_combined = pd.concat(all_data, ignore_index=True)

output_path = '/content/drive/MyDrive/Data Pesanan/Shopee/ShopeeAllOrder.xlsx'
df_combined.to_excel(output_path, index=False)

Ditemukan 6 file di Google Drive. Memulai proses penggabungan...


Cleaning data Shopee

In [4]:
import pandas as pd

try:
    df = pd.read_excel('/content/drive/MyDrive/Data Pesanan/Shopee/ShopeeAllOrder.xlsx')
    print("Data berhasil dimuat.")
except FileNotFoundError:
    print("Error: File 'data.csv' tidak ditemukan. Harap pastikan file berada di direktori yang sama atau berikan path lengkap.")
    df = pd.DataFrame({
        'no. pesanan': ['ORD001', 'ORD002', 'ORD003', 'ORD004', 'ORD005', 'ORD006'],
        'nomor referensi sku': ['AFI-123', 'AFX-456', 'SKU-789', 'AFI-001', 'AFX-002', 'SKU-003'],
        'jumlah': [2, 1, 3, 4, 1, 2],
        'harga setelah diskon': [100.50, 200.75, 50.00, 120.00, 300.50, 75.00],
        'kota/kabupaten': ['Jakarta', 'Surabaya', 'Bandung', 'Medan', 'Makassar', 'Semarang'],
        'provinsi': ['DKI Jakarta', 'Jawa Timur', 'Jawa Barat', 'Sumatera Utara', 'Sulawesi Selatan', 'Jawa Tengah'],
        'kolom lain': ['data', 'lain', 'yang', 'akan', 'dihapus', 'saja']
    })

# Filter Baris
df_filtered = df[df['Nomor Referensi SKU'].str.startswith(('AFI', 'AFX'), na=False)].copy()

# Seleksi Kolom
selected_columns = [
    'No. Pesanan',
    'Nomor Referensi SKU',
    'Jumlah',
    'Harga Setelah Diskon',
    'Kota/Kabupaten',
    'Provinsi'
]
missing_columns = [col for col in selected_columns if col not in df_filtered.columns]
if missing_columns:
    print(f"Peringatan: Kolom berikut tidak ditemukan dan akan diabaikan: {missing_columns}")
    selected_columns = [col for col in selected_columns if col not in missing_columns]
df_cleaned = df_filtered[selected_columns].copy()

# Tambah Kolom 'Nama Poduk'
def get_product_name(sku):
    if sku.startswith('AFI'):
        return 'AFI Stick'
    elif sku.startswith('AFX'):
        return 'AFIXION'
    return None

df_cleaned.loc[:, 'Nama Produk'] = df_cleaned['Nomor Referensi SKU'].apply(get_product_name)

# Pindah Kolom 'Nama Produk'
current_columns = df_cleaned.columns.tolist()
sku_idx = current_columns.index('Nomor Referensi SKU')
if 'Nama Produk' in current_columns:
    current_columns.remove('Nama Produk')
    current_columns.insert(sku_idx + 1, 'Nama Produk')
    df_cleaned = df_cleaned[current_columns]

# Tambah Kolom 'Total Botol'
def calculate_total_bottles(row):
    sku_parts = str(row['Nomor Referensi SKU']).split('-')
    if len(sku_parts) > 1:
        try:
            last_part = sku_parts[-1]
            if last_part == '250' or last_part == '500':
                return row['Jumlah'] * 1
            else:
                return row['Jumlah'] * int(last_part)
        except ValueError:
            return None
    return None

df_cleaned.loc[:, 'Total Botol'] = df_cleaned.apply(calculate_total_bottles, axis=1)

# Pindah Kolom 'Total Botol'
current_columns = df_cleaned.columns.tolist()
jumlah_idx = current_columns.index('Jumlah')
if 'Total Botol' in current_columns:
    current_columns.remove('Total Botol')
    current_columns.insert(jumlah_idx + 1, 'Total Botol')
    df_cleaned = df_cleaned[current_columns]

# Perbaiki format data 'Harga Setelah Diskon'
df_cleaned['Harga Setelah Diskon'] = pd.to_numeric(df_cleaned['Harga Setelah Diskon'], errors='coerce')
df_cleaned['Harga Setelah Diskon'] = df_cleaned['Harga Setelah Diskon'] * 1000
df_cleaned['Harga Setelah Diskon'] = df_cleaned['Harga Setelah Diskon'].apply(lambda x: f"Rp {x:,.2f}" if pd.notna(x) else x)

# Perbaiki format data 'Provinsi' dan 'Kota/Kabupaten'
df_cleaned['Provinsi'] = df_cleaned['Provinsi'].astype(str).str.title()
df_cleaned['Kota/Kabupaten'] = df_cleaned['Kota/Kabupaten'].astype(str).str.title()
prefixes_to_remove = ['Kab.', 'Kabupaten', 'Kota']
for prefix in prefixes_to_remove:
    df_cleaned['Kota/Kabupaten'] = df_cleaned['Kota/Kabupaten'].str.replace(f'^{prefix}\s*', '', regex=True, case=False)

# Tampilkan hasil 5 baris pertama
print("5 baris pertama data yang sudah dibersihkan:")
display(df_cleaned.head())

# Simpan ke file baru bernama 'Shopee-AFI.xlsx' dalam format Excel
df_cleaned.to_excel('/content/drive/MyDrive/Data Pesanan/Shopee/Shopee-AFI.xlsx', index=False)
print("\nData yang sudah dibersihkan telah disimpan ke 'Shopee-AFI.xlsx'")

<>:88: SyntaxWarning: invalid escape sequence '\s'
<>:88: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_11535/2077501037.py:88: SyntaxWarning: invalid escape sequence '\s'
  df_cleaned['Kota/Kabupaten'] = df_cleaned['Kota/Kabupaten'].str.replace(f'^{prefix}\s*', '', regex=True, case=False)


Data berhasil dimuat.
5 baris pertama data yang sudah dibersihkan:


,No. Pesanan,Nomor Referensi SKU,Nama Produk,Jumlah,Total Botol,Harga Setelah Diskon,Kota/Kabupaten,Provinsi
24,2605018JWQQE0Q,AFI-2,AFI Stick,1,2,"Rp 78,774.00",Blitar,Jawa Timur
25,2605018MP608UU,AFI-1,AFI Stick,1,1,"Rp 39,886.00",Malang,Jawa Timur
34,2605018UJN8JTV,AFX-1,AFIXION,1,1,"Rp 148,876.00",Bima,Nusa Tenggara Barat (Ntb)
43,2605018XBKY8GY,AFI-3,AFI Stick,1,3,"Rp 118,161.00",Temanggung,Jawa Tengah
50,2605019182ACY0,AFI-1,AFI Stick,1,1,"Rp 39,886.00",Banjar,Jawa Barat



Data yang sudah dibersihkan telah disimpan ke 'Shopee-AFI.xlsx'


# DATA PESANAN TIKTOK SHOP


Menggabungkan data pesanan Tiktok Shop

In [ ]:
path_folder = '/content/drive/MyDrive/Data Pesanan/Tiktok Shop/Mei 2026'
files = glob.glob(os.path.join(path_folder, "*.xlsx"))

all_data = []

for file in files:
    df = pd.read_excel(file, header=[0, 1])
    all_data.append(df)

df_combined = pd.concat(all_data, ignore_index=True)

df_combined.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df_combined.columns.values]

output_path = '/content/drive/MyDrive/Data Pesanan/Tiktok Shop/Mei 2026/Mei_All_Order.xlsx'
df_combined.to_excel(output_path, index=False)

Cleaning data pesanan Tiktok Shop

In [ ]:
import pandas as pd

# Load Data
try:
    df_tiktok = pd.read_excel('/content/drive/MyDrive/Data Pesanan/Tiktok Shop/Tiktok Shop_All_Order.xlsx')
    print("TikTok data berhasil dimuat.")

except FileNotFoundError:
    print("Error: File 'TikTokAllOrder.xlsx' tidak ditemukan. Harap pastikan file berada di direktori yang benar.")
    df_tiktok = pd.DataFrame({
        'Order ID_Platform unique order ID.': ['TK001', 'TK002', 'TK003', 'TK004', 'TK005'],
        'Quantity_SKU sold quantity in the order.': [1, 2, 1, 3, 1],
        'Seller SKU_Seller sku input by the seller in the product system.': ['AFI-100', 'AFX-500', 'NON-SKU', 'AFI-250', 'AFX-10'],
        'SKU Unit Original Price_1 SKU original price.': [100000, 50000, 20000, 150000, 75000],
        'SKU Seller Discount_Total seller discount in this SKU ID.': [10000, 5000, 0, 15000, 7500],
        'Province_Unnamed: 48_level_1': ['DKI Jakarta', 'Jawa Barat', pd.NA, 'Jawa Timur', pd.NA],
        'Province_Unnamed: 46_level_1': [pd.NA, pd.NA, 'Bali', pd.NA, 'Jawa Tengah'],
        'Regency and City_Unnamed: 49_level_1': ['Jakarta Selatan', 'Bandung', pd.NA, 'Surabaya', pd.NA],
        'Regency and City_Unnamed: 47_level_1': [pd.NA, pd.NA, 'Denpasar', pd.NA, 'Semarang']
    })
    print("Menggunakan DataFrame dummy karena file tidak ditemukan.")

# Filter Baris
df_tiktok_filtered = df_tiktok[df_tiktok['Seller SKU_Seller sku input by the seller in the product system.'].str.startswith(('AFI', 'AFX'), na=False)].copy()

# Seleksi Kolom
selected_columns_tiktok = [
    'Order ID_Platform unique order ID.',
    'Quantity_SKU sold quantity in the order.',
    'Seller SKU_Seller sku input by the seller in the product system.',
    'SKU Unit Original Price_1 SKU original price.',
    'SKU Seller Discount_Total seller discount in this SKU ID.',
    'Province_Unnamed: 48_level_1',
    'Province_Unnamed: 46_level_1',
    'Regency and City_Unnamed: 49_level_1',
    'Regency and City_Unnamed: 47_level_1'
]


actual_tiktok_columns = df_tiktok_filtered.columns.tolist()
missing_tiktok_cols = [col for col in selected_columns_tiktok if col not in actual_tiktok_columns]
if missing_tiktok_cols:
    print(f"Peringatan: Kolom TikTok berikut tidak ditemukan dan akan diabaikan: {missing_tiktok_cols}")
    selected_columns_tiktok = [col for col in selected_columns_tiktok if col not in missing_tiktok_cols]

df_tiktok_cleaned = df_tiktok_filtered[selected_columns_tiktok].copy()

# Menggabungkan Kolom Provinsi dan Kota/Kabupaten
df_tiktok_cleaned['Provinsi'] = df_tiktok_cleaned['Province_Unnamed: 48_level_1'].fillna(df_tiktok_cleaned['Province_Unnamed: 46_level_1'])
df_tiktok_cleaned['Kota/Kabupaten'] = df_tiktok_cleaned['Regency and City_Unnamed: 49_level_1'].fillna(df_tiktok_cleaned['Regency and City_Unnamed: 47_level_1'])

# Drop original location columns
df_tiktok_cleaned = df_tiktok_cleaned.drop(columns=[
    'Province_Unnamed: 48_level_1',
    'Province_Unnamed: 46_level_1',
    'Regency and City_Unnamed: 49_level_1',
    'Regency and City_Unnamed: 47_level_1'
])

# Tambah Kolom 'Nama Produk'
def get_tiktok_product_name(sku):
    if pd.isna(sku):
        return None
    sku_str = str(sku)
    if sku_str.startswith('AFI'):
        return 'AFI Stick'
    elif sku_str.startswith('AFX'):
        return 'AFIXION'
    return None

df_tiktok_cleaned.loc[:, 'Nama Produk'] = df_tiktok_cleaned['Seller SKU_Seller sku input by the seller in the product system.'].apply(get_tiktok_product_name)

# Tambah Kolom 'Total Botol'
def calculate_tiktok_total_bottles(row):
    sku_parts = str(row['Seller SKU_Seller sku input by the seller in the product system.']).split('-')
    if len(sku_parts) > 1:
        try:
            last_part = sku_parts[-1]
            if last_part == '250' or last_part == '500':
                return row['Quantity_SKU sold quantity in the order.'] * 1
            else:
                return row['Quantity_SKU sold quantity in the order.'] * int(last_part)
        except ValueError:
            return None
    return None

df_tiktok_cleaned.loc[:, 'Total Botol'] = df_tiktok_cleaned.apply(calculate_tiktok_total_bottles, axis=1)

# Tambah Kolom 'Omset'
df_tiktok_cleaned['Omset'] = (df_tiktok_cleaned['SKU Unit Original Price_1 SKU original price.'] * df_tiktok_cleaned['Quantity_SKU sold quantity in the order.']) - df_tiktok_cleaned['SKU Seller Discount_Total seller discount in this SKU ID.']
df_tiktok_cleaned['Omset'] = df_tiktok_cleaned['Omset'].apply(lambda x: f"Rp {x:,.2f}" if pd.notna(x) else x)

# Ubah Nama Kolom
df_tiktok_cleaned = df_tiktok_cleaned.rename(columns={
    'Order ID_Platform unique order ID.': 'No. Pesanan',
    'Seller SKU_Seller sku input by the seller in the product system.': 'Nomor Referensi SKU',
    'Quantity_SKU sold quantity in the order.': 'Jumlah',
    'Omset': 'Harga Setelah Diskon'
})

# Hapus Kolom
df_tiktok_cleaned = df_tiktok_cleaned.drop(columns=[
    'SKU Unit Original Price_1 SKU original price.',
    'SKU Seller Discount_Total seller discount in this SKU ID.'
])

# Ubah Urutan Kolom
final_columns_order_tiktok = [
    'No. Pesanan',
    'Nomor Referensi SKU',
    'Nama Produk',
    'Jumlah',
    'Total Botol',
    'Harga Setelah Diskon',
    'Kota/Kabupaten',
    'Provinsi'
]

df_tiktok_cleaned = df_tiktok_cleaned[final_columns_order_tiktok]

# Tampilkan hasil 5 baris pertama
print("5 baris pertama data TikTok yang sudah dibersihkan:")
display(df_tiktok_cleaned.head())

# Simpan ke file baru bernama 'TikTok-AFI.xlsx' dalam format Excel
df_tiktok_cleaned.to_excel('/content/drive/MyDrive/Data Pesanan/Tiktok Shop/TikTok-AFI.xlsx', index=False)
print("\nData TikTok yang sudah dibersihkan telah disimpan ke 'TikTok-AFI.xlsx'")

TikTok data berhasil dimuat.
5 baris pertama data TikTok yang sudah dibersihkan:


,No. Pesanan,Nomor Referensi SKU,Nama Produk,Jumlah,Total Botol,Harga Setelah Diskon,Kota/Kabupaten,Provinsi
1,5.838075e+17,AFI-1,AFI Stick,1.0,1.0,"Rp 39,787.00",Tanah Datar,Sumatera Barat
14,5.838066e+17,AFI-1,AFI Stick,1.0,1.0,"Rp 39,787.00",Kab. Padang Lawas,Sumatera Utara
17,5.838064e+17,AFI-1,AFI Stick,1.0,1.0,"Rp 39,787.00",Kab. Dompu,Nusa Tenggara Barat
24,5.838061e+17,AFI-1,AFI Stick,1.0,1.0,"Rp 39,787.00",Bireuen,Aceh
29,5.838057e+17,AFI-1,AFI Stick,1.0,1.0,"Rp 39,787.00",Kab. Magelang,Jawa Tengah



Data TikTok yang sudah dibersihkan telah disimpan ke 'TikTok-AFI.xlsx'


# PIVOT TABLE

In [1]:
import pandas as pd
import numpy as np

try:
    df = pd.read_excel('/content/drive/MyDrive/Data Pesanan/All Order AFI.xlsx')
except FileNotFoundError:
    data = {
        'Provinsi': ['Jawa Barat', 'Jawa Barat', 'DKI Jakarta', 'DKI Jakarta'],
        'Kota/Kabupaten': ['Bandung', 'Bandung', 'Jakarta Selatan', 'Jakarta Selatan'],
        'Nama Produk': ['AFI Stick', 'AFIXION', 'AFI Stick', 'AFIXION'],
        'Total Botol': [10, 5, 20, 15],
        'Harga Setelah Diskon': [100000, 150000, 200000, 450000]
    }
    df = pd.DataFrame(data)

# Cek Kolom
df['Total Botol'] = pd.to_numeric(df['Total Botol'], errors='coerce').fillna(0)
df['Harga Setelah Diskon'] = pd.to_numeric(df['Harga Setelah Diskon'], errors='coerce').fillna(0)

# Membuat Pivot Table untuk mendapatkan Total Botol dan Harga (Omset) per Produk
pivot_df = pd.pivot_table(
    df,
    values=['Total Botol', 'Harga Setelah Diskon'],
    index=['Provinsi', 'Kota/Kabupaten'],
    columns='Nama Produk',
    aggfunc='sum',
    fill_value=0
)

# Meratakan MultiIndex kolom agar lebih mudah dibaca
pivot_df.columns = [f"{col[1]} ({'Botol' if col[0] == 'Total Botol' else 'Omset'})" for col in pivot_df.columns]

# Menambahkan kolom total keseluruhan
pivot_df['Total Keseluruhan Botol'] = pivot_df.filter(like='(Botol)').sum(axis=1)
pivot_df['Total Keseluruhan Omset'] = pivot_df.filter(like='(Omset)').sum(axis=1)

# Mengurutkan data berdasarkan Total Botol Terjual terbesar
rekap_tabel = pivot_df.sort_values(by='Total Keseluruhan Botol', ascending=False).reset_index()

# Menampilkan output
display(rekap_tabel.head())

,Provinsi,Kota/Kabupaten,AFI Stick (Omset),AFIXION (Omset),AFI Stick (Botol),AFIXION (Botol),Total Keseluruhan Botol,Total Keseluruhan Omset
0,Jawa Timur,Malang,20500919,238021,528,2,530,20738940
1,Jawa Timur,Jember,18574168,148876,479,1,480,18723044
2,Jawa Timur,Banyuwangi,16365898,982401,421,7,428,17348299
3,Jawa Barat,Indramayu,14963454,514078,391,4,395,15477532
4,Jawa Timur,Bojonegoro,14822686,1295591,378,11,389,16118277
